## Lab 10

John is on his way back home for his family dinner. <br>
He is already running late and he’s trying to determine whether he should drive or run with a shortcut path. <br>
It takes 10 minutes to get to his house by walking. <br>
There is a chance that driving may be delayed by traffic lights for 2, 4, or 5 minutes, <br> 
each with a probability of 1/3. Driving might take a longer route that takes 6 minutes without waiting for traffic lights. <br>

The utility for arriving home is a function of t, the number of minutes it takes him to get home, is <br>
                                                𝑈(𝑡)={ 0, 𝑡≤0   <br>
                                                       (2𝑡),𝑡>0

In [18]:
import clips 
import logging

# Setup working environment
logging.basicConfig(level=logging.INFO,format='%(message)s')
    
env = clips.Environment()
router = clips.LoggingRouter()
env.add_router(router)

env.build("""
(deftemplate option
   (slot name))
""")

env.build("""
(deftemplate outcome
   (slot option-name)
   (slot time)
   (slot probability))
""")

env.build("""
(deftemplate expected-utility
   (slot option-name)
   (slot value))
   """)

env.build("""
(deffunction U (?t)
   (if (<= ?t 0) then
      (return 0)
   else
      (return (* 2 ?t))))
      """)

env.build("""
(defrule evaluate-options
   ?opt <- (option (name ?n))
   =>
   (bind ?EU 0)
   ; loop over all outcomes linked to this option
   (do-for-all-facts ((?o outcome)) (eq ?o:option-name ?n)
      (bind ?EU (+ ?EU (* ?o:probability (U ?o:time)))))
   (assert (expected-utility (option-name ?n) (value ?EU)))
   (printout t "Expected utility of " ?n " = " ?EU crlf))
""")

env.build("""
(defrule choose-best
   ?e1 <- (expected-utility (option-name ?n1) (value ?v1))
   ?e2 <- (expected-utility (option-name ?n2&~?n1) (value ?v2))
   (not (expected-utility (value ?v3&:(> ?v3 ?v1))))
   =>
   (printout t "Best choice is " ?n1 " with expected utility " ?v1 crlf))
""")

env.eval("(assert (option (name walk)))")
env.eval("(assert (outcome (option-name walk) (time 10) (probability 1.0)))")

env.eval("(assert (option (name drive)))")
env.eval("(assert (outcome (option-name drive) (time 8) (probability 0.333)))")
env.eval("(assert (outcome (option-name drive) (time 10) (probability 0.333)))")
env.eval("(assert (outcome (option-name drive) (time 11) (probability 0.333)))")

env.run()



Expected utility of drive = 19.314
Expected utility of walk = 20.0
Best choice is walk with expected utility 20.0


3